# Velocity Profile and Integral Length Scale vs Terrain Category

This notebook compares the simulated streamwise velocity profile and the integral length scale of the streamwise component (`ux`) extracted from a line probe against a single, user-selected EN 1991-1-4 (Eurocode) terrain category.

- Mean velocity profile `u(z)/u_ref` plotted against the chosen EU category.
- Turbulence intensity `Iu(z)` plotted against the same category.
- Integral length scale `Lu(z) = u_mean * T_int`, where `T_int` is the integral of the temporal autocorrelation up to the first zero crossing, plotted against the EN 1991-1-4 model `L(z) = Lt * (z/zt)^alpha`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load data

In [ ]:
points_path = "fixtures/line.line_profile line.points.csv"
ux_path = "fixtures/line.line_profile line.ux.csv"

df_pts = pd.read_csv(points_path, index_col="idx")
df_ux = pd.read_csv(ux_path)
time_steps = df_ux["time_step"].to_numpy(dtype=np.float64)
df_ux.drop(columns=("time_step"), inplace=True)

df_ux.columns = df_ux.columns.astype(int)

z = df_pts["z"].to_numpy()
u_mean = df_ux.mean(axis=0).to_numpy()
u_rms = df_ux.std(axis=0).to_numpy()

dt = time_steps[1] - time_steps[0]
fs = 1.0 / dt

print(f"Points shape: {df_pts.shape}")
print(f"Velocity shape: {df_ux.shape}  (time steps x probe points)")
print(f"z range: {z.min():.1f} - {z.max():.1f} m")
print(f"dt = {dt:.6f} s  ->  fs = {fs:.3f} Hz")

## EN 1991-1-4 terrain categories

Each category is defined by a roughness length `z0` and a minimum height `z_min`.
The mean velocity profile is `u(z)/u_ref = kr * ln(z/z0)`, with `kr = 0.19 * (z0/z0_II)^0.07` and `z0_II = 0.05 m`.
Turbulence intensity is `Iu(z) = 1 / ln(z/z0)` (with `kI = c0 = 1`).
Integral length scale is `L(z) = Lt * (z/zt)^alpha` with `Lt = 300 m`, `zt = 200 m`, `alpha = 0.67 + 0.05 * ln(z0)`; clamped to `L(z_min)` for `z < z_min`.

In [ ]:
# EN 1991-1-4 Table 4.1 (Eurocode terrain categories)
EU_CATEGORIES = {
    "Cat 0": {"z0": 0.003, "z_min": 1.0},
    "Cat I": {"z0": 0.01, "z_min": 1.0},
    "Cat II": {"z0": 0.05, "z_min": 2.0},
    "Cat III": {"z0": 0.30, "z_min": 5.0},
    "Cat IV": {"z0": 1.00, "z_min": 10.0},
}

Z0_REF = 0.05  # z0_II
L_T = 300.0  # reference integral length [m]
Z_T = 200.0  # reference height [m]


def eu_velocity_profile(z_arr, z0):
    """Normalized mean velocity u(z)/u_ref per EN 1991-1-4 (with c0 = 1, c_prob = 1)."""
    kr = 0.19 * (z0 / Z0_REF) ** 0.07
    return kr * np.log(np.maximum(z_arr, 1e-9) / z0)


def eu_turbulence_intensity(z_arr, z0, z_min):
    """Iu(z) per EN 1991-1-4, clamped to z_min."""
    z_eff = np.maximum(z_arr, z_min)
    return 1.0 / np.log(z_eff / z0)


def eu_integral_length(z_arr, z0, z_min):
    """L(z) per EN 1991-1-4 Annex B.1, clamped to L(z_min) for z < z_min."""
    alpha = 0.67 + 0.05 * np.log(z0)
    z_eff = np.maximum(z_arr, z_min)
    return L_T * (z_eff / Z_T) ** alpha

## Select terrain category

Set `category` to one of: `Cat 0`, `Cat I`, `Cat II`, `Cat III`, `Cat IV`.

In [ ]:
category = "Cat II"

if category not in EU_CATEGORIES:
    raise ValueError(f"Unknown category {category!r}; choose one of {list(EU_CATEGORIES)}")

cat_params = EU_CATEGORIES[category]
print(f"Selected: {category}  |  z0 = {cat_params['z0']} m  |  z_min = {cat_params['z_min']} m")

## Velocity profile vs category

In [ ]:
u_ref = 20.0  # reference velocity used to normalize the simulation [m/s]
z_eu = np.linspace(0.5, 200, 500)

SIM_COLOR = "#e69f00"
REF_COLOR = "#000000"
GRID_COLOR = "#cccccc"
LW = 2.0

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_facecolor("#ffffff")
ax.grid(True, color=GRID_COLOR, linewidth=0.8, linestyle="-")

ax.plot(u_mean / u_ref, z, color=SIM_COLOR, linewidth=LW, label="AeroSim LES")
ax.plot(
    eu_velocity_profile(z_eu, cat_params["z0"]),
    z_eu,
    color=REF_COLOR,
    linewidth=LW,
    linestyle="--",
    label=f"EN 1991-1-4 {category}",
)
ax.set_xlabel("u_mean / u_ref [-]", fontsize=12)
ax.set_ylabel("z [m]", fontsize=12)
ax.set_title("Mean longitudinal velocity (0-200 m)", fontsize=13)
ax.set_ylim(2, 200)
ax.tick_params(labelsize=11)
ax.legend(fontsize=10, loc="lower right")

fig.tight_layout()
plt.show()

## Turbulence intensity vs category

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_facecolor("#ffffff")
ax.grid(True, color=GRID_COLOR, linewidth=0.8, linestyle="-")

ax.plot(u_rms / u_mean, z, color=SIM_COLOR, linewidth=LW, label="AeroSim LES")
ax.plot(
    eu_turbulence_intensity(z_eu, cat_params["z0"], cat_params["z_min"]),
    z_eu,
    color=REF_COLOR,
    linewidth=LW,
    linestyle="--",
    label=f"EN 1991-1-4 {category}",
)
ax.set_xlabel("Iu [-]", fontsize=12)
ax.set_ylabel("z [m]", fontsize=12)
ax.set_title("Turbulence intensity (0-200 m)", fontsize=13)
ax.set_ylim(2, 200)
ax.set_xlim(0, 0.6)
ax.tick_params(labelsize=11)
ax.legend(fontsize=10, loc="upper right")

fig.tight_layout()
plt.show()

## Integral length scale from data

Following Taylor's frozen-turbulence hypothesis: `Lu = u_mean * T_int`, where

    T_int = integral_0^tau* R_uu(tau) dtau

with `R_uu(tau)` the normalized autocorrelation of the fluctuating component `u'(t) = u(t) - u_mean`, and `tau*` the first zero crossing of `R_uu`. Computed via FFT for efficiency.

In [ ]:
def autocorrelation_fft(signal: np.ndarray) -> np.ndarray:
    """Normalized autocorrelation R_uu(tau) for tau >= 0 via FFT."""
    s = signal - signal.mean()
    n = s.size
    nfft = 1 << (2 * n - 1).bit_length()
    F = np.fft.rfft(s, n=nfft)
    acf_full = np.fft.irfft(F * np.conj(F), n=nfft)[:n]
    norm = np.arange(n, 0, -1)
    acf = acf_full / norm
    return acf / acf[0]


def integral_length_scale(signal: np.ndarray, dt: float, u_mean: float) -> float:
    """Lu = u_mean * integral of R_uu(tau) up to the first zero crossing."""
    acf = autocorrelation_fft(signal)
    sign_change = np.where(np.diff(np.sign(acf)))[0]
    if sign_change.size == 0:
        idx_zero = acf.size
    else:
        idx_zero = sign_change[0] + 1
    T_int = np.trapz(acf[:idx_zero], dx=dt)
    return u_mean * T_int

In [ ]:
Lu_sim = np.zeros(df_pts.shape[0])
for i, point_idx in enumerate(df_pts.index):
    u = df_ux[point_idx].to_numpy(dtype=float)
    if u.std() < 1e-9:
        Lu_sim[i] = np.nan
        continue
    Lu_sim[i] = integral_length_scale(u, dt=dt, u_mean=u.mean())

for z_target in [10, 20, 50, 100]:
    j = int((np.abs(z - z_target)).argmin())
    print(f"z = {z[j]:>6.2f} m  ->  Lu = {Lu_sim[j]:7.2f} m")

## Integral length scale vs EU values

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_facecolor("#ffffff")
ax.grid(True, color=GRID_COLOR, linewidth=0.8, linestyle="-")

ax.plot(Lu_sim, z, color=SIM_COLOR, linewidth=LW, label="AeroSim LES")
ax.plot(
    eu_integral_length(z_eu, cat_params["z0"], cat_params["z_min"]),
    z_eu,
    color=REF_COLOR,
    linewidth=LW,
    linestyle="--",
    label=f"EN 1991-1-4 {category}",
)

ax.set_xlabel("Lu [m]", fontsize=12)
ax.set_ylabel("z [m]", fontsize=12)
ax.set_title("Streamwise integral length scale (0-200 m)", fontsize=13)
ax.set_ylim(2, 200)
ax.set_xlim(left=0)
ax.legend(fontsize=10, loc="lower right")
ax.tick_params(labelsize=11)

fig.tight_layout()
plt.show()